In [1]:
# ==============================================================================
# CELL 1: IMPORTS & SETUP
# ==============================================================================

import os
import json
import time
import warnings
import numpy as np

import torch
import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.callbacks import BaseCallback

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from numba import njit

warnings.filterwarnings("ignore")

# ---- Device setup (MacBook, 18GB, Apple Silicon MPS) ----
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"[✓] Using device: {DEVICE}")

# CHANGED: new folder — keeps this board's checkpoint/VecNormalize stats
# completely separate from the old 18-component run's (different obs/action
# dims -> loading the old ones here would crash or silently corrupt training).
OUTPUT_DIR = "optimized_layouts_rl_29comp"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

[✓] Using device: mps


In [6]:
# ==============================================================================
# CELL 2: LOAD CONFIG & BUILD COMPONENT ARRAYS
# ==============================================================================

def load_config(filepath='newobj.json'):
    if not os.path.exists(filepath):
        filepath = os.path.join(os.getcwd(), 'newobj.json')
    if not os.path.exists(filepath):
        raise FileNotFoundError("❌ Unable to locate 'newobj.json' in working directory.")
    with open(filepath, 'r') as f:
        data = json.load(f)
    print(f"[✓] Config loaded: '{filepath}'")
    return data

data = load_config('newobj.json')

# ---- Board parameters ----
panel_width    = float(data['BOARD']['Length (mm)'])
panel_height   = float(data['BOARD']['Breadth (mm)'])
border_spacing = float(data['BOARD']['Border (mm)'])
clearance      = float(data['BOARD']['Clearance (mm)'])

# ---- Flatten FRONT + BACK components, expanding Object qty > 1 ----
raw_components_to_process = []
for comp in data['COMPONENTS'].get('FRONT', []):
    raw_components_to_process.append((comp, False))
for comp in data['COMPONENTS'].get('BACK', []):
    raw_components_to_process.append((comp, True))

components_to_process = []
for comp, is_back in raw_components_to_process:
    qty = int(comp.get('Object qty', 1))
    if qty > 1:
        for q in range(qty):
            cloned = comp.copy()
            cloned['Unique Name'] = f"{comp['Object name']}{q}"
            components_to_process.append((cloned, is_back))
    else:
        cloned = comp.copy()
        cloned['Unique Name'] = comp['Object name']
        components_to_process.append((cloned, is_back))

num_elements = len(components_to_process)

element_names        = []
element_shapes        = []
is_back_side_list     = []
flat_offsets_list      = []
clearance_dirs_list    = []
element_data           = []  # [mass, length, width, insert_diam]

offset_counts  = np.zeros(num_elements, dtype=np.int32)
offset_indices = np.zeros(num_elements, dtype=np.int32)
current_idx = 0

for idx, (component, is_back) in enumerate(components_to_process):
    element_name = component['Unique Name']
    shape = component.get('Shape', 'rectangle')
    element_shapes.append(shape)
    is_back_side_list.append(1 if is_back else 0)

    # Custom face clearances (CF)
    cf_faces = component.get('CF', [])
    cf_len = float(component.get('CFLen (mm)', clearance))
    c_dirs = [clearance, clearance, clearance, clearance]  # [Top, Right, Bottom, Left]
    for face in cf_faces:
        if face == 1: c_dirs[0] = cf_len
        elif face == 2: c_dirs[1] = cf_len
        elif face == 3: c_dirs[2] = cf_len
        elif face == 4: c_dirs[3] = cf_len
    clearance_dirs_list.append(c_dirs)

    if shape == 'rectangle':
        length = float(component['Length (mm)'])
        width  = float(component['Breadth (mm)'])
        insert_diam = float(component['Insert (mm)'])
        insert_qty  = int(component.get('Insert qty', 4))
        half_l, half_w = length / 2.0, width / 2.0

        if insert_qty == 2:
            offsets = [(half_l, 0.0), (-half_l, 0.0)] if length >= width else [(0.0, half_w), (0.0, -half_w)]
        elif insert_qty == 6:
            offsets = [(half_l, half_w), (half_l, -half_w), (-half_l, half_w), (-half_l, -half_w)]
            offsets.extend([(0.0, half_w), (0.0, -half_w)] if length >= width else [(half_l, 0.0), (-half_l, 0.0)])
        else:
            offsets = [(half_l, half_w), (half_l, -half_w), (-half_l, half_w), (-half_l, -half_w)]
    else:
        diameter = float(component['Diameter (mm)'])
        length, width = diameter, diameter
        insert_diam = float(component['Insert (mm)'])
        insert_qty  = int(component.get('Insert qty', 3))
        inner_radius = max(0.0, (diameter / 2.0) - 1.0)
        offsets = [(inner_radius * np.cos(2*np.pi*k/insert_qty), inner_radius * np.sin(2*np.pi*k/insert_qty))
                   for k in range(insert_qty)]

    flat_offsets_list.extend(offsets)
    offset_counts[idx] = len(offsets)
    offset_indices[idx] = current_idx
    current_idx += len(offsets)

    element_data.append([float(component['Weight (kg)']), length, width, insert_diam])
    element_names.append(element_name)

flat_offsets   = np.array(flat_offsets_list, dtype=np.float64)
element_data   = np.array(element_data, dtype=np.float64)
masses         = element_data[:, 0]
lengths        = element_data[:, 1]
widths         = element_data[:, 2]
insert_diams   = element_data[:, 3]
is_back_side   = np.array(is_back_side_list, dtype=np.int32)
clearance_dirs = np.array(clearance_dirs_list, dtype=np.float64)
n = num_elements

# ---- Required pin-to-pin distance matrix (by insert size) ----
req_d_matrix = np.full((n, n), 30.0)
for i in range(n):
    for j in range(n):
        if insert_diams[i] == 4.0 and insert_diams[j] == 4.0:
            req_d_matrix[i, j] = 24.0
        elif insert_diams[i] == 6.0 and insert_diams[j] == 6.0:
            req_d_matrix[i, j] = 36.0

print(f"[✓] {n} components parsed ({int(is_back_side.sum())} back-side, {n - int(is_back_side.sum())} front-side)")
for i, nm in enumerate(element_names):
    print(f"    {i:2d}. {nm:8s} | {element_shapes[i]:9s} | mass={masses[i]:.3f}kg | pins={offset_counts[i]}")

[✓] Config loaded: 'newobj.json'
[✓] 30 components parsed (0 back-side, 30 front-side)
     0. Bat10    | rectangle | mass=37.500kg | pins=4
     1. Bat20    | rectangle | mass=37.500kg | pins=4
     2. IRU      | rectangle | mass=13.000kg | pins=6
     3. OBC10    | rectangle | mass=16.000kg | pins=4
     4. OBC20    | rectangle | mass=16.000kg | pins=4
     5. ICP      | rectangle | mass=14.900kg | pins=4
     6. SPDU     | rectangle | mass=9.000kg | pins=4
     7. FDM10    | rectangle | mass=3.000kg | pins=6
     8. FDM20    | rectangle | mass=3.000kg | pins=6
     9. IAPM1    | rectangle | mass=14.000kg | pins=6
    10. IAPM2    | rectangle | mass=14.000kg | pins=6
    11. IAPR1    | rectangle | mass=14.000kg | pins=6
    12. IAPR2    | rectangle | mass=14.000kg | pins=6
    13. BMU51    | rectangle | mass=1.500kg | pins=4
    14. BMU52    | rectangle | mass=1.500kg | pins=4
    15. BMU53    | rectangle | mass=1.500kg | pins=4
    16. BMU54    | rectangle | mass=1.500kg | pins=4
  

In [7]:
# ==============================================================================
# CELL 3: COORDINATE BOUNDS + NUMBA PENALTY-BREAKDOWN ENGINE
# ==============================================================================

# ---- Per-component (x_min,x_max,y_min,y_max) bounds so its footprint+clearance+pins
#      can never physically leave the active board area ----
lower_bounds = []
upper_bounds = []
for i in range(n):
    i_start = offset_indices[i]
    i_count = offset_counts[i]
    comp_offsets = flat_offsets[i_start : i_start + i_count]

    max_off_x = max(abs(x) for x, _ in comp_offsets) if i_count > 0 else 0.0
    max_off_y = max(abs(y) for _, y in comp_offsets) if i_count > 0 else 0.0

    c_top   = clearance_dirs[i, 0]
    c_right = clearance_dirs[i, 3] if is_back_side[i] else clearance_dirs[i, 1]
    c_bot   = clearance_dirs[i, 2]
    c_left  = clearance_dirs[i, 1] if is_back_side[i] else clearance_dirs[i, 3]

    req_margin_left   = max(lengths[i]/2.0 + c_left,  max_off_x + insert_diams[i]/2.0)
    req_margin_right  = max(lengths[i]/2.0 + c_right, max_off_x + insert_diams[i]/2.0)
    req_margin_bottom = max(widths[i]/2.0 + c_bot,    max_off_y + insert_diams[i]/2.0)
    req_margin_top    = max(widths[i]/2.0 + c_top,    max_off_y + insert_diams[i]/2.0)

    if not is_back_side[i]:
        x_min = -panel_width/2.0 + border_spacing + req_margin_left
        x_max =  panel_width/2.0 - border_spacing - req_margin_right
    else:
        x_min = -panel_width/2.0 + border_spacing + req_margin_right
        x_max =  panel_width/2.0 - border_spacing - req_margin_left

    y_min = -panel_height/2.0 + border_spacing + req_margin_bottom
    y_max =  panel_height/2.0 - border_spacing - req_margin_top

    lower_bounds.extend([x_min, y_min])
    upper_bounds.extend([x_max, y_max])

lower_bounds = np.array(lower_bounds, dtype=np.float64)
upper_bounds = np.array(upper_bounds, dtype=np.float64)

# Sanity check: bounds must not be inverted (component too big for board)
bad = np.where(lower_bounds > upper_bounds)[0]
if len(bad) > 0:
    print(f"[!] WARNING: {len(bad)} bound(s) inverted — component may not physically fit. Check board size / component sizes.")
else:
    print("[✓] All per-component coordinate bounds are valid.")

CG_TOLERANCE = 1.0  # mm

@njit(fastmath=True, cache=True)
def compute_penalty_breakdown(cx, cy, masses, hl, hw, req_d_matrix, num_components,
                               flat_offsets, offset_counts, offset_indices, is_back_side,
                               clearance_dirs, panel_width, panel_height, border_spacing):
    """
    Same geometric rules as the CMA-ES fitness core, but returns the 4 penalty
    components SEPARATELY (border, overlap, insert, cg) instead of one summed score,
    so the RL reward can weight priority-1 constraints vs priority-2 CG differently.
    """
    total_mass = 0.0
    cg_x = 0.0
    cg_y = 0.0
    abs_x = np.zeros(num_components)

    for i in range(num_components):
        m = masses[i]
        total_mass += m
        abs_x[i] = -cx[i] if is_back_side[i] else cx[i]
        cg_x += abs_x[i] * m
        cg_y += cy[i] * m

    if total_mass == 0.0:
        return 1e15, 1e15, 1e15, 1e15

    cg_x /= total_mass
    cg_y /= total_mass

    active_x_min = -panel_width/2.0 + border_spacing
    active_x_max =  panel_width/2.0 - border_spacing
    active_y_min = -panel_height/2.0 + border_spacing
    active_y_max =  panel_height/2.0 - border_spacing

    cg_offset = np.sqrt(cg_x**2 + cg_y**2)
    cg_penalty = 0.0
    if cg_offset > CG_TOLERANCE:
        cg_penalty = (cg_offset - CG_TOLERANCE) ** 2

    overlap_penalty = 0.0
    border_penalty = 0.0

    for i in range(num_components):
        c_top   = clearance_dirs[i, 0]
        c_right = clearance_dirs[i, 3] if is_back_side[i] else clearance_dirs[i, 1]
        c_bot   = clearance_dirs[i, 2]
        c_left  = clearance_dirs[i, 1] if is_back_side[i] else clearance_dirs[i, 3]

        i_x_min = abs_x[i] - hl[i] - c_left
        i_x_max = abs_x[i] + hl[i] + c_right
        i_y_min = cy[i] - hw[i] - c_bot
        i_y_max = cy[i] + hw[i] + c_top

        viol_left   = max(0.0, active_x_min - i_x_min)
        viol_right  = max(0.0, i_x_max - active_x_max)
        viol_bottom = max(0.0, active_y_min - i_y_min)
        viol_top    = max(0.0, i_y_max - active_y_max)

        total_border_viol = viol_left + viol_right + viol_bottom + viol_top
        if total_border_viol > 0.0:
            border_penalty += total_border_viol ** 2

        for j in range(i + 1, num_components):
            if is_back_side[i] == is_back_side[j]:
                cj_top   = clearance_dirs[j, 0]
                cj_right = clearance_dirs[j, 3] if is_back_side[j] else clearance_dirs[j, 1]
                cj_bot   = clearance_dirs[j, 2]
                cj_left  = clearance_dirs[j, 1] if is_back_side[j] else clearance_dirs[j, 3]

                j_x_min = abs_x[j] - hl[j] - cj_left
                j_x_max = abs_x[j] + hl[j] + cj_right
                j_y_min = cy[j] - hw[j] - cj_bot
                j_y_max = cy[j] + hw[j] + cj_top

                overlap_x = max(0.0, min(i_x_max, j_x_max) - max(i_x_min, j_x_min))
                overlap_y = max(0.0, min(i_y_max, j_y_max) - max(i_y_min, j_y_min))

                if overlap_x > 0.0 and overlap_y > 0.0:
                    overlap_area = overlap_x * overlap_y
                    overlap_penalty += overlap_area ** 2

    insert_penalty = 0.0
    for i in range(num_components):
        i_start = offset_indices[i]
        i_count = offset_counts[i]
        for j in range(i + 1, num_components):
            j_start = offset_indices[j]
            j_count = offset_counts[j]
            req_dist = req_d_matrix[i, j]

            for ii in range(i_count):
                idx_i = i_start + ii
                xi_abs = abs_x[i] + (-flat_offsets[idx_i, 0] if is_back_side[i] else flat_offsets[idx_i, 0])
                yi_abs = cy[i] + flat_offsets[idx_i, 1]

                for jj in range(j_count):
                    idx_j = j_start + jj
                    xj_abs = abs_x[j] + (-flat_offsets[idx_j, 0] if is_back_side[j] else flat_offsets[idx_j, 0])
                    yj_abs = cy[j] + flat_offsets[idx_j, 1]

                    dist = np.sqrt((xi_abs - xj_abs)**2 + (yi_abs - yj_abs)**2)
                    if req_dist > dist:
                        insert_penalty += (req_dist - dist) ** 2

    return border_penalty, overlap_penalty, insert_penalty, cg_penalty


def evaluate_layout(cx, cy):
    """Convenience wrapper: takes full coordinate arrays, returns penalty breakdown dict."""
    hl = lengths / 2.0
    hw = widths / 2.0
    b, o, ins, cg = compute_penalty_breakdown(
        cx, cy, masses, hl, hw, req_d_matrix, n,
        flat_offsets, offset_counts, offset_indices, is_back_side,
        clearance_dirs, panel_width, panel_height, border_spacing
    )
    return {"border": b, "overlap": o, "insert": ins, "cg": cg}


# Quick smoke test: evaluate at bounds midpoint (should still have some overlap most likely)
mid_cx = (lower_bounds[0::2] + upper_bounds[0::2]) / 2.0
mid_cy = (lower_bounds[1::2] + upper_bounds[1::2]) / 2.0
test_result = evaluate_layout(mid_cx, mid_cy)
print("[✓] Numba fitness engine compiled & tested. Sample penalty breakdown at bounds-midpoint layout:")
for k, v in test_result.items():
    print(f"    {k:8s}: {v:,.2f}")

[✓] All per-component coordinate bounds are valid.
[✓] Numba fitness engine compiled & tested. Sample penalty breakdown at bounds-midpoint layout:
    border  : 0.00
    overlap : 1,467,450,506,142.00
    insert  : 142,257.95
    cg      : 188.01


In [8]:
# ==============================================================================
# CELL 4: GYMNASIUM ENVIRONMENT (FINAL — annealed step, cx/cy in info to avoid
# SubprocVecEnv auto-reset race)
# ==============================================================================

MAX_STEP_MM   = 12.0     # nudge size at the START of an episode (coarse untangling)
MIN_STEP_MM   = 0.25     # nudge size by the END of an episode (fine insert-distance precision)
MAX_STEPS     = 250       # episode horizon (truncation)
STEP_COST     = 0.01      # tiny per-step penalty to encourage speed
SUCCESS_BONUS = 50.0

# Insert weighted highest (border/overlap already solvable easily); CG stays lowest (priority 2)
SCORE_WEIGHTS = {"border": 3.0, "overlap": 3.0, "insert": 10.0, "cg": 0.5}
VALID_EPS = 1e-6  # below this, a penalty term counts as "resolved"


def composite_score(breakdown):
    """Weighted sum of log1p-compressed penalty terms. Lower = better, 0 = perfect."""
    return (SCORE_WEIGHTS["border"]  * np.log1p(breakdown["border"]) +
            SCORE_WEIGHTS["overlap"] * np.log1p(breakdown["overlap"]) +
            SCORE_WEIGHTS["insert"]  * np.log1p(breakdown["insert"]) +
            SCORE_WEIGHTS["cg"]      * np.log1p(breakdown["cg"]))


def is_fully_valid(breakdown):
    return (breakdown["border"] < VALID_EPS and
            breakdown["overlap"] < VALID_EPS and
            breakdown["insert"] < VALID_EPS)


class PCBPlacementEnv(gym.Env):
    """
    Iterative-refinement placement environment.
    Episode: start from a random layout within per-component bounds.
    Each step: agent nudges every component's (x,y) simultaneously.
    Nudge size is ANNEALED within the episode: MAX_STEP_MM early (fast overlap
    resolution) -> MIN_STEP_MM late (fine insert-distance / CG tuning).
    Reward: reduction in composite penalty score (dense), + terminal bonus if fully valid.

    IMPORTANT: cx/cy are embedded directly into `info` at the end of step().
    Do NOT rely on a separate post-hoc state accessor (e.g. calling back into
    the env after the fact) to read cx/cy for "the layout that just finished" —
    SubprocVecEnv auto-resets a sub-env internally the instant an episode
    terminates/truncates, so by the time any external call reaches the worker
    process, self.cx/self.cy already belong to the NEXT episode. Reading cx/cy
    out of `info` inside this same step() call is the only race-free way.
    """
    metadata = {"render_modes": []}

    def __init__(self, seed=None):
        super().__init__()
        self.n = n
        self.lower = lower_bounds.astype(np.float32)
        self.upper = upper_bounds.astype(np.float32)
        self.range_ = (self.upper - self.lower)
        self.range_[self.range_ == 0] = 1.0

        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(2 * self.n,), dtype=np.float32)
        obs_dim = 2 * self.n + 4 + 1
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)

        self.cx = None
        self.cy = None
        self.step_count = 0
        self._np_random_seed = seed

    def _normalize_pos(self):
        nx = 2.0 * (self.cx - self.lower[0::2]) / self.range_[0::2] - 1.0
        ny = 2.0 * (self.cy - self.lower[1::2]) / self.range_[1::2] - 1.0
        return np.stack([nx, ny], axis=1).ravel().astype(np.float32)

    def _get_breakdown(self):
        b, o, ins, cg = compute_penalty_breakdown(
            self.cx, self.cy, masses, lengths/2.0, widths/2.0, req_d_matrix, self.n,
            flat_offsets, offset_counts, offset_indices, is_back_side,
            clearance_dirs, panel_width, panel_height, border_spacing
        )
        return {"border": b, "overlap": o, "insert": ins, "cg": cg}

    def _get_obs(self, breakdown):
        pos = self._normalize_pos()
        pen = np.array([
            np.log1p(breakdown["border"]),
            np.log1p(breakdown["overlap"]),
            np.log1p(breakdown["insert"]),
            np.log1p(breakdown["cg"]),
        ], dtype=np.float32)
        step_frac = np.array([self.step_count / MAX_STEPS], dtype=np.float32)
        return np.concatenate([pos, pen, step_frac])

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        rng = self.np_random
        self.cx = rng.uniform(self.lower[0::2], self.upper[0::2]).astype(np.float32)
        self.cy = rng.uniform(self.lower[1::2], self.upper[1::2]).astype(np.float32)
        self.step_count = 0

        breakdown = self._get_breakdown()
        self.prev_score = composite_score(breakdown)
        obs = self._get_obs(breakdown)
        info = {"breakdown": breakdown}
        return obs, info

    def step(self, action):
        action = np.clip(action, -1.0, 1.0).astype(np.float32)

        progress = self.step_count / MAX_STEPS
        current_max_step = MAX_STEP_MM * (1.0 - progress) + MIN_STEP_MM * progress

        dx = action[0::2] * current_max_step
        dy = action[1::2] * current_max_step

        self.cx = np.clip(self.cx + dx, self.lower[0::2], self.upper[0::2])
        self.cy = np.clip(self.cy + dy, self.lower[1::2], self.upper[1::2])
        self.step_count += 1

        breakdown = self._get_breakdown()
        score = composite_score(breakdown)

        reward = (self.prev_score - score) - STEP_COST
        self.prev_score = score

        terminated = False
        if is_fully_valid(breakdown):
            reward += SUCCESS_BONUS
            terminated = True

        truncated = self.step_count >= MAX_STEPS
        obs = self._get_obs(breakdown)

        # cx/cy captured HERE, same process/same call, before any auto-reset can occur
        info = {"breakdown": breakdown, "score": score, "cx": self.cx.copy(), "cy": self.cy.copy()}

        return obs, float(reward), terminated, truncated, info


# ---- Sanity check ----
_test_env = PCBPlacementEnv(seed=SEED)
check_env(_test_env, warn=True)
print("[✓] PCBPlacementEnv passed gymnasium/SB3 env checker.")

obs, info = _test_env.reset(seed=SEED)
print(f"[✓] Reset OK. obs shape={obs.shape}, initial breakdown={ {k: round(v,2) for k,v in info['breakdown'].items()} }")
total_r = 0.0
for _ in range(5):
    a = _test_env.action_space.sample()
    obs, r, term, trunc, info = _test_env.step(a)
    total_r += r
print(f"[✓] 5 random steps ran fine. cumulative reward={total_r:.3f}, has cx/cy in info: {'cx' in info and 'cy' in info}")

[✓] PCBPlacementEnv passed gymnasium/SB3 env checker.
[✓] Reset OK. obs shape=(65,), initial breakdown={'border': 0.0, 'overlap': 15645790202.91, 'insert': 490.83, 'cg': 87370.82}
[✓] 5 random steps ran fine. cumulative reward=2.480, has cx/cy in info: True


In [9]:
# ==============================================================================
# CELL 5: VECTORIZED ENVIRONMENTS (envs only — model is built/loaded in Cell 6)
# ==============================================================================

from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.monitor import Monitor

N_ENVS = 8

def make_env(rank, seed=SEED):
    def _init():
        env = PCBPlacementEnv(seed=seed + rank)
        env = Monitor(env)
        return env
    return _init

vec_env = SubprocVecEnv([make_env(i) for i in range(N_ENVS)])

VECNORM_PATH = os.path.join(OUTPUT_DIR, "vecnormalize.pkl")
if os.path.exists(VECNORM_PATH):
    vec_env = VecNormalize.load(VECNORM_PATH, vec_env)
    print(f"[✓] Loaded existing VecNormalize stats from '{VECNORM_PATH}'.")
else:
    vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0, clip_reward=10.0, gamma=0.99)
    print("[✓] Created fresh VecNormalize (no existing stats found).")

vec_env.training = True
vec_env.norm_reward = True

print(f"[✓] {N_ENVS} parallel environments ready (SubprocVecEnv + VecNormalize).")

[✓] Created fresh VecNormalize (no existing stats found).
[✓] 8 parallel environments ready (SubprocVecEnv + VecNormalize).


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [15]:
# ==============================================================================
# CELL 6: MODEL (LOAD-OR-CREATE) + FIXED CALLBACK + TRAINING
# ==============================================================================

MODEL_PATH = os.path.join(OUTPUT_DIR, "ppo_pcb_placement")

policy_kwargs = dict(
    net_arch=dict(pi=[256, 256], vf=[256, 256]),
    activation_fn=torch.nn.Tanh,
)

if os.path.exists(MODEL_PATH + ".zip"):
    model = PPO.load(MODEL_PATH, env=vec_env, device=DEVICE)
    print(f"[✓] Loaded existing checkpoint from '{MODEL_PATH}.zip'.")
else:
    model = PPO(
        policy="MlpPolicy",
        env=vec_env,
        device=DEVICE,
        learning_rate=3e-4,
        n_steps=1024,
        batch_size=256,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.005,
        vf_coef=0.5,
        max_grad_norm=0.5,
        policy_kwargs=policy_kwargs,
        verbose=1,
        seed=SEED,
        tensorboard_log=os.path.join(OUTPUT_DIR, "tb_logs"),
    )
    print("[✓] Created fresh PPO model (no existing checkpoint found).")


class BestLayoutCallback(BaseCallback):
    """
    Tracks best fully-valid layout (lowest CG penalty) and best invalid layout
    (lowest composite score) across all parallel envs. cx/cy are read directly
    from info — populated inside env.step() BEFORE any SubprocVecEnv auto-reset
    can overwrite them. Do not fetch state via any other channel.
    """
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.best_valid_score = np.inf
        self.best_valid_layout = None
        self.best_invalid_score = np.inf
        self.best_invalid_layout = None
        self.n_valid_found = 0

    def _on_step(self) -> bool:
        infos = self.locals.get("infos", [])
        for info in infos:
            breakdown = info.get("breakdown")
            if breakdown is None:
                continue
            score = composite_score(breakdown)
            if is_fully_valid(breakdown):
                self.n_valid_found += 1
                if breakdown["cg"] < self.best_valid_score:
                    self.best_valid_score = breakdown["cg"]
                    self.best_valid_layout = {"cx": info["cx"], "cy": info["cy"], "breakdown": dict(breakdown)}
            else:
                if score < self.best_invalid_score:
                    self.best_invalid_score = score
                    self.best_invalid_layout = {"cx": info["cx"], "cy": info["cy"], "breakdown": dict(breakdown)}

        if self.n_calls % 2000 == 0:
            self.logger.record("custom/best_valid_cg", self.best_valid_score if self.best_valid_layout else -1)
            self.logger.record("custom/best_invalid_score", self.best_invalid_score)
            self.logger.record("custom/n_valid_found", self.n_valid_found)
        return True


best_layout_cb = BestLayoutCallback(verbose=1)

TRAIN_TIMESTEPS = 6_000_000  # bump this and re-run the cell anytime to keep training the same checkpoint

print(f"[▶] Training for {TRAIN_TIMESTEPS:,} timesteps on device='{DEVICE}'...")
t0 = time.time()

model.learn(
    total_timesteps=TRAIN_TIMESTEPS,
    callback=best_layout_cb,
    progress_bar=True,
    reset_num_timesteps=False,
)

elapsed = time.time() - t0
print(f"\n[✓] Training complete in {elapsed/60:.1f} minutes.")
print(f"[✓] Fully valid layouts found this run: {best_layout_cb.n_valid_found}")
if best_layout_cb.best_valid_layout:
    print(f"[✓] Best valid layout CG penalty: {best_layout_cb.best_valid_score:.4f}")
    print(f"    Full breakdown: {best_layout_cb.best_valid_layout['breakdown']}")
else:
    print(f"[!] No fully valid layout this run. Best invalid composite score: {best_layout_cb.best_invalid_score:.4f}")
    print(f"    Breakdown: {best_layout_cb.best_invalid_layout['breakdown']}")

model.save(MODEL_PATH)
vec_env.save(VECNORM_PATH)
print(f"[✓] Model and VecNormalize stats saved to '{OUTPUT_DIR}/'.")

[✓] Loaded existing checkpoint from 'optimized_layouts_rl_29comp/ppo_pcb_placement.zip'.
[▶] Training for 6,000,000 timesteps on device='mps'...
Logging to optimized_layouts_rl_29comp/tb_logs/PPO_0
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 250      |
|    ep_rew_mean     | -0.994   |
| time/              |          |
|    fps             | 3723     |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 7028736  |
---------------------------------
-----------------------------------------
| custom/                 |             |
|    best_invalid_score   | 73.6        |
|    best_valid_cg        | -1          |
|    n_valid_found        | 0           |
| rollout/                |             |
|    ep_len_mean          | 250         |
|    ep_rew_mean          | -0.83       |
| time/                   |             |
|    fps                  | 2511        |
|    iterations           | 2           |



[✓] Training complete in 50.4 minutes.
[✓] Fully valid layouts found this run: 0
[!] No fully valid layout this run. Best invalid composite score: 68.5331
    Breakdown: {'border': 0.0, 'overlap': 1557519736.1635725, 'insert': 0.0, 'cg': 23575.576867079828}
[✓] Model and VecNormalize stats saved to 'optimized_layouts_rl_29comp/'.


In [16]:
model.save(MODEL_PATH)
vec_env.save(VECNORM_PATH)
print(f"Saved at {model.num_timesteps} timesteps")

Saved at 13025280 timesteps
